# Cross-lingual Hybrid Retrieval with Haystack

When building search systems for multilingual content, a common challenge arises: **keyword-based retrieval (BM25) only matches documents in the same language as the query**, while **dense retrieval with multilingual embeddings can bridge languages but may miss exact term matches**.

This cookbook demonstrates how to build a **hybrid retrieval pipeline** in Haystack that combines BM25 and multilingual dense embeddings to handle cross-lingual search effectively. We'll work with a mixed Chinese-English document collection and show how hybrid retrieval outperforms either method alone.

**What you'll learn:**
- How BM25 fails on cross-lingual queries
- How multilingual dense embeddings enable cross-lingual retrieval
- How to combine both with Haystack's `DocumentJoiner` using Reciprocal Rank Fusion
- How to add a cross-encoder re-ranker for even better results

> 💡 **Real-world motivation:** In e-commerce platforms serving multiple markets, product descriptions may exist in different languages. A user searching in English should still find relevant Chinese-language product pages, and vice versa. This pattern applies broadly to multilingual knowledge bases, academic literature, and enterprise document search.

## Install Dependencies

We'll use Haystack's in-memory document store (no external database needed) with `sentence-transformers` for multilingual embeddings.

In [26]:
%%bash
pip install -q haystack-ai "sentence-transformers>=3.0.0" "transformers[torch,sentencepiece]"

## Prepare a Multilingual Document Collection

Let's create a small but realistic dataset of documents in both English and Chinese. These cover overlapping topics — renewable energy, AI policy, and urban planning — so we can test whether retrieval works **across language boundaries**.

In [27]:
from haystack import Document

documents = [
    # English documents
    Document(
        content="Solar panel efficiency has improved significantly in recent years. "
                "Modern photovoltaic cells can convert over 22% of sunlight into electricity, "
                "making rooftop solar installations increasingly cost-effective for homeowners.",
        meta={"language": "en", "topic": "renewable_energy"}
    ),
    Document(
        content="The European Union has set ambitious carbon neutrality targets for 2050. "
                "Key policies include the Emissions Trading System, renewable energy mandates, "
                "and substantial funding for green hydrogen research.",
        meta={"language": "en", "topic": "climate_policy"}
    ),
    Document(
        content="Large language models are transforming how developers write code. "
                "AI-assisted programming tools can suggest entire functions, detect bugs, "
                "and explain complex codebases, significantly boosting productivity.",
        meta={"language": "en", "topic": "ai_programming"}
    ),
    Document(
        content="Urban green spaces play a crucial role in reducing heat island effects. "
                "Cities that invest in parks, green roofs, and tree-lined streets see measurable "
                "improvements in air quality and residents' mental health.",
        meta={"language": "en", "topic": "urban_planning"}
    ),
    Document(
        content="Wind power capacity reached record levels globally in 2024. "
                "Offshore wind farms in particular are becoming major contributors "
                "to national energy grids across Europe and East Asia.",
        meta={"language": "en", "topic": "renewable_energy"}
    ),
    # Chinese documents
    Document(
        content="中国的碳中和目标要求在2060年前实现净零排放。"
                "主要措施包括大规模发展光伏发电、推进电动汽车普及、"
                "以及建立全国性的碳排放交易市场。",
        meta={"language": "zh", "topic": "climate_policy"}
    ),
    Document(
        content="深度学习技术在自然语言处理领域取得了重大突破。"
                "基于Transformer架构的大语言模型能够理解上下文语义，"
                "在机器翻译、文本摘要和代码生成等任务上表现优异。",
        meta={"language": "zh", "topic": "ai_programming"}
    ),
    Document(
        content="城市热岛效应是现代都市面临的重要环境问题。"
                "通过增加城市绿化面积、推广绿色屋顶和透水路面，"
                "可以有效降低城区温度并改善居民生活环境。",
        meta={"language": "zh", "topic": "urban_planning"}
    ),
    Document(
        content="新型钙钛矿太阳能电池的转换效率已突破25%。"
                "与传统硅基电池相比，钙钛矿电池制造成本更低，"
                "柔性基底的特性使其可以应用于建筑外墙和便携设备。",
        meta={"language": "zh", "topic": "renewable_energy"}
    ),
    Document(
        content="检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。"
                "系统首先从文档集合中检索相关段落，然后将检索结果作为上下文"
                "输入给生成模型，从而减少幻觉问题并提供可追溯的信息来源。",
        meta={"language": "zh", "topic": "rag"}
    ),
]

print(f"Total documents: {len(documents)}")
print(f"English: {sum(1 for d in documents if d.meta['language'] == 'en')}")
print(f"Chinese: {sum(1 for d in documents if d.meta['language'] == 'zh')}")

Total documents: 10
English: 5
Chinese: 5


## Approach 1: BM25 Retrieval (Keyword-based)

Let's first try BM25, the classic keyword-matching algorithm. BM25 works by matching exact terms between the query and documents. This works well for **monolingual** search but has an obvious limitation: it cannot match terms across languages.

Let's see what happens when we query in English against our mixed-language collection.

In [28]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# Create document store and write documents
bm25_store = InMemoryDocumentStore()
bm25_store.write_documents(documents)

bm25_retriever = InMemoryBM25Retriever(document_store=bm25_store, top_k=5)

# Test: English query about carbon neutrality
query = "carbon neutrality policies and emission targets"
results = bm25_retriever.run(query=query)

print(f"Query: '{query}'")
print(f"Results found: {len(results['documents'])}\n")
for i, doc in enumerate(results["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

Query: 'carbon neutrality policies and emission targets'
Results found: 5

  [1] (lang=en, score=9.3893)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=en, score=6.1257)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [3] (lang=en, score=5.9469)
      Large language models are transforming how developers write code. AI-assisted pr...

  [4] (lang=en, score=5.9372)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [5] (lang=en, score=5.5397)
      Solar panel efficiency has improved significantly in recent years. Modern photov...



**Observation:** BM25 only retrieves the English document about EU carbon neutrality. It completely misses the Chinese document about China's carbon neutrality goals — even though it's highly relevant.

Now let's try the reverse: a Chinese query.

In [29]:
# Test: Chinese query about solar energy
query_zh = "太阳能电池效率提升"
results_zh = bm25_retriever.run(query=query_zh)

print(f"Query: '{query_zh}' (solar cell efficiency improvement)")
print(f"Results found: {len(results_zh['documents'])}\n")
for i, doc in enumerate(results_zh["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

Query: '太阳能电池效率提升' (solar cell efficiency improvement)
Results found: 0



**Observation:** BM25 finds only the Chinese perovskite solar cell document. The English document about solar panel efficiency — covering essentially the same topic — is invisible to keyword matching because the terms don't overlap across languages.

## Approach 2: Dense Retrieval (Multilingual Embeddings)

Now let's use a **multilingual embedding model** that maps both Chinese and English text into a shared semantic vector space. We'll use `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, which supports 50+ languages.

The key insight: semantically similar content in different languages gets mapped to nearby vectors, enabling cross-lingual retrieval.

In [30]:
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Embed and index all documents
dense_store = InMemoryDocumentStore()
doc_embedder = SentenceTransformersDocumentEmbedder(model=EMBEDDING_MODEL)
doc_embedder.warm_up()
docs_with_embeddings = doc_embedder.run(documents=documents)
dense_store.write_documents(docs_with_embeddings["documents"])

print(f"Indexed {dense_store.count_documents()} documents with embeddings")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 10 documents with embeddings


In [31]:
# Dense retrieval with the same English query
text_embedder = SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
text_embedder.warm_up()

dense_retriever = InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)

query = "carbon neutrality policies and emission targets"
query_embedding = text_embedder.run(text=query)
results = dense_retriever.run(query_embedding=query_embedding["embedding"])

print(f"Query: '{query}'")
print(f"Results found: {len(results['documents'])}\n")
for i, doc in enumerate(results["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'carbon neutrality policies and emission targets'
Results found: 5

  [1] (lang=en, score=18.3640)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=zh, score=14.1291)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...

  [3] (lang=en, score=7.1499)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [4] (lang=en, score=6.3158)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [5] (lang=en, score=5.8462)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...



**Key result:** Dense retrieval with multilingual embeddings successfully retrieves **both** the English EU policy document **and** the Chinese carbon neutrality document. The embedding model understands that "carbon neutrality" and "碳中和" are semantically equivalent.

## Approach 3: Hybrid Retrieval Pipeline

While dense retrieval handles cross-lingual matching, BM25 still has strengths: it's great at matching **exact terms, names, and technical identifiers** that embedding models might overlook. The best approach is to combine both methods.

Haystack's `DocumentJoiner` merges results from multiple retrievers using **Reciprocal Rank Fusion (RRF)**, which combines ranked lists without requiring score normalization.

The formula:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

where $r(d)$ is the rank of document $d$ in retriever $r$, and $k$ is a constant (default 60).

In [32]:
from haystack import Pipeline
from haystack.components.joiners import DocumentJoiner
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever,
)
from haystack.components.embedders import SentenceTransformersTextEmbedder

# We need a single document store that has both content and embeddings
# dense_store already has embeddings, and InMemoryDocumentStore supports BM25 natively

hybrid_pipeline = Pipeline()

# Components
hybrid_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
)
hybrid_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=dense_store, top_k=5)
)
hybrid_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)
)
hybrid_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion", top_k=5)
)

# Connections
hybrid_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
hybrid_pipeline.connect("bm25_retriever.documents", "joiner.documents")
hybrid_pipeline.connect("embedding_retriever.documents", "joiner.documents")

print(hybrid_pipeline)

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - bm25_retriever: InMemoryBM25Retriever
  - embedding_retriever: InMemoryEmbeddingRetriever
  - joiner: DocumentJoiner
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query_embedding (list[float])
  - bm25_retriever.documents -> joiner.documents (list[Document])
  - embedding_retriever.documents -> joiner.documents (list[Document])



Now let's run the hybrid pipeline with our cross-lingual test queries:

In [33]:
def run_hybrid_query(pipeline, query: str, label: str = ""):
    """Run a query through the hybrid pipeline and display results."""
    results = pipeline.run({
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query},
    })
    
    docs = results["joiner"]["documents"]
    if label:
        print(f"{'='*60}")
        print(f"{label}")
    print(f"Query: '{query}'")
    print(f"Results: {len(docs)}\n")
    for i, doc in enumerate(docs):
        print(f"  [{i+1}] (lang={doc.meta['language']}, topic={doc.meta['topic']}, score={doc.score:.4f})")
        print(f"      {doc.content[:80]}...\n")
    return docs


# Test 1: English query — should find both EN and ZH carbon neutrality docs
run_hybrid_query(
    hybrid_pipeline,
    "carbon neutrality policies and emission targets",
    label="Test 1: English query on a topic covered in both languages"
)

# Test 2: Chinese query — should find both ZH and EN solar energy docs
run_hybrid_query(
    hybrid_pipeline,
    "太阳能电池效率提升",
    label="Test 2: Chinese query on a topic covered in both languages"
)

# Test 3: English query with specific technical term
run_hybrid_query(
    hybrid_pipeline,
    "RAG retrieval augmented generation reduce hallucination",
    label="Test 3: English query matching a Chinese-only document"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 1: English query on a topic covered in both languages
Query: 'carbon neutrality policies and emission targets'
Results: 5

  [1] (lang=en, topic=climate_policy, score=1.0000)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=en, topic=urban_planning, score=0.9761)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [3] (lang=en, topic=renewable_energy, score=0.9458)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [4] (lang=en, topic=renewable_energy, score=0.9458)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [5] (lang=zh, topic=climate_policy, score=0.4919)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 2: Chinese query on a topic covered in both languages
Query: '太阳能电池效率提升'
Results: 5

  [1] (lang=en, topic=renewable_energy, score=0.5000)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [2] (lang=zh, topic=renewable_energy, score=0.4919)
      新型钙钛矿太阳能电池的转换效率已突破25%。与传统硅基电池相比，钙钛矿电池制造成本更低，柔性基底的特性使其可以应用于建筑外墙和便携设备。...

  [3] (lang=zh, topic=climate_policy, score=0.4841)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...

  [4] (lang=en, topic=renewable_energy, score=0.4766)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [5] (lang=en, topic=urban_planning, score=0.4692)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 3: English query matching a Chinese-only document
Query: 'RAG retrieval augmented generation reduce hallucination'
Results: 5

  [1] (lang=zh, topic=rag, score=1.0000)
      检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。系统首先从文档集合中检索相关段落，然后将检索结果作为上下文输入给生成模型，从而减少幻觉问题并提...

  [2] (lang=en, topic=renewable_energy, score=0.9685)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [3] (lang=en, topic=ai_programming, score=0.9607)
      Large language models are transforming how developers write code. AI-assisted pr...

  [4] (lang=zh, topic=ai_programming, score=0.4919)
      深度学习技术在自然语言处理领域取得了重大突破。基于Transformer架构的大语言模型能够理解上下文语义，在机器翻译、文本摘要和代码生成等任务上表现优异。...

  [5] (lang=en, topic=climate_policy, score=0.4841)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...



[Document(id=8d4b9e2c1b56afc9ce9ce799bfc695d47111f8c164d928a92dc6579f0d9f462b, content: '检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。系统首先从文档集合中检索相关段落，然后将检索结果作为上下文输入给生成模型，从而减少幻觉问题并提供可追溯的信息来源。', meta: {'language': 'zh', 'topic': 'rag'}, score: 1.0),
 Document(id=731aedac5a0d1a057fbbd4ca5c24bd61f616ed5d362bab73d36942e344479434, content: 'Solar panel efficiency has improved significantly in recent years. Modern photovoltaic cells can con...', meta: {'language': 'en', 'topic': 'renewable_energy'}, score: 0.9684979838709676),
 Document(id=ab23724d86ff952513d5abd131db4e3bf178a437b7f2691657288c814b7ec22c, content: 'Large language models are transforming how developers write code. AI-assisted programming tools can ...', meta: {'language': 'en', 'topic': 'ai_programming'}, score: 0.9606894841269841),
 Document(id=fa543a8d3e47c5bd991c42592866bb25e9ac127266f3744be2edcef5c695beff, content: '深度学习技术在自然语言处理领域取得了重大突破。基于Transformer架构的大语言模型能够理解上下文语义，在机器翻译、文本摘要和代码生成等任务上表现优异。', meta: {'language': 'zh', 'topic': 'ai_p

## Comparing the Three Approaches

Let's systematically compare how each method handles cross-lingual queries:

In [34]:
def count_languages(docs):
    """Count documents by language."""
    langs = {}
    for doc in docs:
        lang = doc.meta.get("language", "unknown")
        langs[lang] = langs.get(lang, 0) + 1
    return langs


test_queries = [
    ("carbon neutrality policies", "碳中和政策"),
    ("solar cell efficiency", "太阳能电池效率"),
    ("urban heat island green spaces", "城市热岛效应 绿化"),
    ("AI code generation LLM", "AI代码生成 大语言模型"),
]

print(f"{'Query (EN)':<35} {'Method':<12} {'EN docs':<10} {'ZH docs':<10}")
print("-" * 67)

for query_en, query_zh in test_queries:
    # BM25 only
    bm25_results = bm25_retriever.run(query=query_en)["documents"]
    bm25_langs = count_languages(bm25_results)
    
    # Dense only
    q_emb = text_embedder.run(text=query_en)
    dense_results = dense_retriever.run(query_embedding=q_emb["embedding"])["documents"]
    dense_langs = count_languages(dense_results)
    
    # Hybrid
    hybrid_results = hybrid_pipeline.run({
        "text_embedder": {"text": query_en},
        "bm25_retriever": {"query": query_en},
    })["joiner"]["documents"]
    hybrid_langs = count_languages(hybrid_results)
    
    q_display = query_en[:33]
    print(f"{q_display:<35} {'BM25':<12} {bm25_langs.get('en', 0):<10} {bm25_langs.get('zh', 0):<10}")
    print(f"{'':35} {'Dense':<12} {dense_langs.get('en', 0):<10} {dense_langs.get('zh', 0):<10}")
    print(f"{'':35} {'Hybrid':<12} {hybrid_langs.get('en', 0):<10} {hybrid_langs.get('zh', 0):<10}")
    print()

Query (EN)                          Method       EN docs    ZH docs   
-------------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

carbon neutrality policies          BM25         5          0         
                                    Dense        3          2         
                                    Hybrid       4          1         



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

urban heat island green spaces      BM25         5          0         
                                    Dense        3          2         
                                    Hybrid       4          1         



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

AI code generation LLM              BM25         5          0         
                                    Dense        2          3         
                                    Hybrid       4          1         



The results show that:
- **BM25** retrieves only same-language documents (EN → EN, ZH → ZH)
- **Dense retrieval** finds relevant documents across both languages
- **Hybrid** combines both strengths: it captures cross-lingual semantic matches while still rewarding exact term overlap

## (Optional) Adding a Cross-Encoder Re-Ranker

For production systems, you can further improve result quality by adding a **cross-encoder re-ranker**. Cross-encoders process the query and document together (rather than independently), enabling more accurate relevance judgments.

We'll use `SentenceTransformersSimilarityRanker` with the `cross-encoder/ms-marco-MiniLM-L-6-v2` model. This is the recommended ranker in Haystack 2.x, supporting PyTorch, ONNX, and OpenVINO backends. For a fully multilingual re-ranker, consider `nreimers/mmarco-mMiniLMv2-L12-H384-v1`.

In [35]:
from haystack.components.rankers import SentenceTransformersSimilarityRanker

# Build hybrid pipeline with re-ranker
hybrid_rerank_pipeline = Pipeline()

hybrid_rerank_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
)
hybrid_rerank_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=dense_store, top_k=5)
)
hybrid_rerank_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)
)
hybrid_rerank_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion", top_k=10)
)
hybrid_rerank_pipeline.add_component(
    "ranker",
    SentenceTransformersSimilarityRanker(
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k=5
    )
)

# Connections
hybrid_rerank_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
hybrid_rerank_pipeline.connect("bm25_retriever.documents", "joiner.documents")
hybrid_rerank_pipeline.connect("embedding_retriever.documents", "joiner.documents")
hybrid_rerank_pipeline.connect("joiner.documents", "ranker.documents")

print(hybrid_rerank_pipeline)

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - bm25_retriever: InMemoryBM25Retriever
  - embedding_retriever: InMemoryEmbeddingRetriever
  - joiner: DocumentJoiner
  - ranker: SentenceTransformersSimilarityRanker
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query_embedding (list[float])
  - bm25_retriever.documents -> joiner.documents (list[Document])
  - embedding_retriever.documents -> joiner.documents (list[Document])
  - joiner.documents -> ranker.documents (list[Document])



In [36]:
# Run the re-ranked hybrid pipeline
query = "carbon neutrality policies and emission targets"
results = hybrid_rerank_pipeline.run({
    "text_embedder": {"text": query},
    "bm25_retriever": {"query": query},
    "ranker": {"query": query},
})

print(f"Query: '{query}'")
print(f"Re-ranked results:\n")
for i, doc in enumerate(results["ranker"]["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, topic={doc.meta['topic']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


README.md: 0.00B [00:00, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'carbon neutrality policies and emission targets'
Re-ranked results:

  [1] (lang=en, topic=climate_policy, score=0.9997)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=zh, topic=climate_policy, score=0.0000)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...

  [3] (lang=en, topic=ai_programming, score=0.0000)
      Large language models are transforming how developers write code. AI-assisted pr...

  [4] (lang=en, topic=urban_planning, score=0.0000)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [5] (lang=en, topic=renewable_energy, score=0.0000)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...



## Building a Full Cross-lingual RAG Pipeline

Finally, let's connect our hybrid retriever to a generator to build a complete cross-lingual RAG pipeline. The generator can synthesize information from documents in both languages to provide a comprehensive answer.

In [37]:
from haystack.components.builders import PromptBuilder

template = """
You are a helpful multilingual assistant. Answer the question based on the provided
context documents, which may be in English or Chinese. Synthesize information from
all relevant documents regardless of their language. Answer in the same language as
the question.

Context:
{% for doc in documents %}
[{{ doc.meta.language | upper }}] {{ doc.content }}
{% endfor %}

Question: {{ query }}
Answer:
"""

prompt_builder = PromptBuilder(template=template)

# Build the prompt to inspect what the generator would receive
query = "What are the main approaches to carbon neutrality in different regions?"
hybrid_results = hybrid_pipeline.run({
    "text_embedder": {"text": query},
    "bm25_retriever": {"query": query},
})

prompt = prompt_builder.run(
    query=query,
    documents=hybrid_results["joiner"]["documents"]
)

print("Generated prompt (truncated):")
print(prompt["prompt"][:500])
print("...")
print(f"\nTotal prompt length: {len(prompt['prompt'])} characters")
print("\n💡 To complete this RAG pipeline, connect the prompt_builder to any")
print("   Haystack generator (OpenAI, Anthropic, HuggingFace, etc.).")
print("   See: https://docs.haystack.deepset.ai/docs/generators")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated prompt (truncated):

You are a helpful multilingual assistant. Answer the question based on the provided
context documents, which may be in English or Chinese. Synthesize information from
all relevant documents regardless of their language. Answer in the same language as
the question.

Context:

[EN] The European Union has set ambitious carbon neutrality targets for 2050. Key policies include the Emissions Trading System, renewable energy mandates, and substantial funding for green hydrogen research.

[EN] Urban gr
...

Total prompt length: 1280 characters

💡 To complete this RAG pipeline, connect the prompt_builder to any
   Haystack generator (OpenAI, Anthropic, HuggingFace, etc.).
   See: https://docs.haystack.deepset.ai/docs/generators


## Key Takeaways

1. **BM25 alone is language-blind.** It only matches documents sharing the same script and vocabulary as the query. For multilingual collections, this means you miss potentially half your relevant content.

2. **Multilingual dense embeddings bridge languages.** Models like `paraphrase-multilingual-MiniLM-L12-v2` map semantically similar content into nearby vectors regardless of language, enabling true cross-lingual search.

3. **Hybrid retrieval combines the best of both.** BM25 catches exact keyword matches (important for technical terms, names, identifiers), while dense retrieval handles semantic and cross-lingual matching. Reciprocal Rank Fusion merges both result sets without requiring score normalization.

4. **Cross-encoder re-rankers refine results.** Adding a re-ranker on top of hybrid retrieval further improves ranking quality, especially for nuanced relevance judgments.

5. **This pattern generalizes.** The same approach works for any language pair supported by the embedding model — Arabic-English, Japanese-German, Spanish-French, etc.

## 📚 Resources

- [Haystack Hybrid Retrieval Tutorial](https://haystack.deepset.ai/tutorials/33_hybrid_retrieval)
- [Hybrid Retrieval with BM42 — Cookbook](https://haystack.deepset.ai/cookbook/hybrid_retrieval_bm42)
- [Sparse Embedding Retrieval with SPLADE — Cookbook](https://haystack.deepset.ai/cookbook/sparse_embedding_retrieval)
- [Sentence-Transformers Multilingual Models](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html#multilingual-models)
- [DocumentJoiner API Reference](https://docs.haystack.deepset.ai/docs/documentjoiner)
- [TransformersRanker API Reference](https://docs.haystack.deepset.ai/docs/transformersranker)